In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/Intent-Classification-ML-Project/

/content/drive/MyDrive/Intent-Classification-ML-Project


In [ ]:
!ls


data  notebooks  README.md  requirements.txt  src


# Loading Stored Dataset with v1 features

In [ ]:
import pandas as pd

# Paths should match what you used in 01
features_path = "./data/rba_features_v1.parquet"

df = pd.read_parquet(features_path)

print("Loaded shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())  # first 40 cols, just to confirm

df.head()

Loaded shape: (300000, 57)

Columns:
['Login Timestamp', 'User ID', 'Round-Trip Time [ms]', 'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String', 'Browser Name and Version', 'OS Name and Version', 'Device Type', 'Login Successful', 'Is Attack IP', 'Is Account Takeover', 'browser', 'os', 'hour', 'dayofweek', 'is_new_device_for_user', 'is_new_ip_for_user', 'is_off_hours', 'failed_login', 'failures_last_5', 'failure_streak', 'failure_streak_capped', 'location', 'new_location_flag', 'new_asn_flag', 'device_fingerprint', 'valid_device', 'new_device_flag', 'device_change_rate', 'ts_sec', 'delta_sec', 'new_window', 'window_id', 'logins_5min', 'failure_flag', 'burst_failure_count', 'failure_rate', 'streak_reset', 'streak_id', 'failure_streak_length', 'location_freq', 'location_rarity', 'device_type_freq', 'device_type_rarity', 'asn_freq', 'asn_rarity', 'location_rarity_q', 'device_type_rarity_q', 'asn_rarity_q', 'user_offhour_rate', 'is_unusual_time_for_user', 'user_hour_std',

,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,...,device_type_rarity,asn_freq,asn_rarity,location_rarity_q,device_type_rarity_q,asn_rarity_q,user_offhour_rate,is_unusual_time_for_user,user_hour_std,user_hour_std_q
0,2020-02-06 17:10:54.364,-9223287066183308537,541.0,84.209.76.159,no,oslo county,oslo,41164,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...,Chrome 69.0.3497.17.19,...,0.000013,15067,0.000066,1,0,0,0.0,0,0.0,0
1,2020-02-06 19:52:41.530,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0.000005,36,0.027778,0,0,2,0.0,0,0.5,0
2,2020-02-06 20:55:19.627,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0.000005,36,0.027778,0,0,2,0.0,0,0.5,0
3,2020-02-05 21:03:20.657,-9223200578825105501,541.0,79.161.56.83,no,vestfold og telemark,holmestrand,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Chrome Mobile 81.0.4044.2033,...,0.000005,86455,0.000012,3,0,0,0.0,0,0.0,0
4,2020-02-06 19:12:29.501,-9223199305075633823,541.0,79.161.86.86,no,-,-,29695,Mozilla/5.0 (Linux; U; Android 13.0; i phone X...,Opera Mobile 52.1.2254,...,0.000005,86455,0.000012,0,0,0,0.0,0,0.0,0


# Define the binary rule indicators (Iᵢ)



> rule_unusual_time – off-hours & unusual for this user
	2.	rule_off_hours – off-hours, but not unusual (user sometimes uses nights)
	3.	rule_new_device – device changed since last login
	4.	rule_new_asn – ASN changed since last login
	5.	rule_recent_failures – at least 3 failures in last 5 logins



In [ ]:
# --- 1) Rule: Unusual time for this user ---
# 1 if this login is off-hours AND user rarely uses off-hours (< 0.2),
# we already encoded this as is_unusual_time_for_user
df["rule_unusual_time"] = df["is_unusual_time_for_user"].astype(int)

# --- 2) Rule: Off-hours (but NOT already counted as unusual) ---
# Example: user sometimes logs at night, so it's off-hours but not rare for them
df["rule_off_hours"] = (
    (df["is_off_hours"] == 1) &
    (df["is_unusual_time_for_user"] == 0)
).astype(int)

# --- 3) Rule: New device since last login ---
df["rule_new_device"] = df["new_device_flag"].astype(int)

# --- 4) Rule: New ASN since last login ---
df["rule_new_asn"] = df["new_asn_flag"].astype(int)

# --- 5) Rule: Recent failures: 3 or more in previous 5 logins ---
# Make sure NaNs in failures_last_5 are treated as 0
failures_last_5_clean = df["failures_last_5"].fillna(0)
df["rule_recent_failures"] = (failures_last_5_clean >= 3).astype(int)

# Quick sanity check: show first few columns
df[[
    "is_off_hours",
    "is_unusual_time_for_user",
    "failures_last_5",
    "rule_unusual_time",
    "rule_off_hours",
    "rule_new_device",
    "rule_new_asn",
    "rule_recent_failures"
]].head()

,is_off_hours,is_unusual_time_for_user,failures_last_5,rule_unusual_time,rule_off_hours,rule_new_device,rule_new_asn,rule_recent_failures
0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0


Validating the Distributions just to make sure not all are 0's or 1's. We want evenly distributed!

In [ ]:
rule_cols = [
    "rule_unusual_time",
    "rule_off_hours",
    "rule_new_device",
    "rule_new_asn",
    "rule_recent_failures",
]

for col in rule_cols:
    print(f"\n{col} value counts:")
    print(df[col].value_counts())


rule_unusual_time value counts:
rule_unusual_time
0    282476
1     17524
Name: count, dtype: int64

rule_off_hours value counts:
rule_off_hours
0    278226
1     21774
Name: count, dtype: int64

rule_new_device value counts:
rule_new_device
0    184842
1    115158
Name: count, dtype: int64

rule_new_asn value counts:
rule_new_asn
0    201323
1     98677
Name: count, dtype: int64

rule_recent_failures value counts:
rule_recent_failures
0    189779
1    110221
Name: count, dtype: int64


Compute rule_risk_score


In [ ]:
# Compute rule-based risk score as weighted sum of rule indicators

df["rule_risk_score"] = (
    2 * df["rule_unusual_time"]
    + 1 * df["rule_off_hours"]
    + 1 * df["rule_new_device"]
    + 1 * df["rule_new_asn"]
    + 1 * df["rule_recent_failures"]
)

print("rule_risk_score summary:")
print(df["rule_risk_score"].describe())

print("\nCounts per rule_risk_score:")
print(df["rule_risk_score"].value_counts().sort_index())

rule_risk_score summary:
count    300000.000000
mean          1.269593
std           1.537271
min           0.000000
25%           0.000000
50%           0.000000
75%           3.000000
max           5.000000
Name: rule_risk_score, dtype: float64

Counts per rule_risk_score:
rule_risk_score
0    154037
1     37065
2     13555
3     78883
4      2246
5     14214
Name: count, dtype: int64



	•	Median = 0 → for at least 50% of logins, none of the 5 rules fired.
	•	These are “totally clean” from the baseline’s point of view.
	•	75th percentile = 3 → the top 25% of logins have score ≥ 3, meaning:
	•	multiple rules triggered together (e.g., off-hours + new device + failures).
	•	Mean ~1.27, std ~1.54 → most logins are low-to-moderate risk; a smaller group has piled-up risk.

This is good:
We don’t want everything to look risky, but we do want a clear separation between “no rule triggered” and “multiple rules triggered”.

🔹 Score = 0 → 154,037 logins (~51.3%)à

	•	None of the rules fired:
	•	Not off-hours
	•	Not unusual time
	•	No new device
	•	No new ASN
	•	No heavy recent failures

👉 These are your “baseline-normal” logins.
This is exactly what we want: about half the data looks totally unremarkable.

⸻

🔹 Score = 1 → 37,065 logins (~12.4%)
	•	Exactly one mild rule fired:
	•	Example: just off-hours but not unusual for that user,
	•	or just new device,
	•	or just new ASN,
	•	or just 3+ recent failures.

👉 These are slightly risky but not alarming.
Good place to label as low risk but “watch”.

⸻

🔹 Score = 2 → 13,555 logins (~4.5%)
	•	Either:
	•	Unusual time only (2 points), or
	•	two mild rules (e.g., new device + new ASN).

👉 This is your first “this looks genuinely suspicious” layer.

Remember: unusual time on its own got +2 because in EDA it had ~2× attack rate.

⸻

🔹 Score = 3 → 78,883 logins (~26.3%)
	•	Examples:
	•	unusual time (2) + new device (1)
	•	unusual time (2) + recent failures (1)
	•	or 3 different +1 rules together (off-hours + new ASN + recent failures, etc.)

👉 This is a big chunk of the dataset (~1/4) where multiple risk factors align.

This is the region where “step-up auth” is very natural: user might still be legit, but we want an extra check.

⸻

🔹 Score = 4 → 2,246 logins (~0.75%)
	•	Example:
	•	unusual time (2) + any two of {off-hours, new device, new ASN, recent failures}
	•	or four mild rules firing at the same time.

👉 Very clustered risk; these are strong candidates for:
	•	Step-up auth at minimum, often block / review.

⸻

🔹 Score = 5 → 14,214 logins (~4.7%)
	•	This is near-max risk:
	•	E.g., unusual time (2) + three other risk factors (off-hours + new device + recent failures), etc.

👉 These are your top-risk logins.
In a real system, many of these would probably be outright blocked or heavily challenged.

# Potential extra rule signals (for v2 baseline later)
	1.	Device type risk
	•	From EDA: Device Type = bot and unknown had very high attack rates.
	•	Rule idea:
	•	rule_risky_device_type = 1 if device_type in {bot, unknown}
	2.	Global rarity
	•	asn_rarity or location_rarity:
	•	Very rare ASNs or locations might be suspicious.
	•	Rule idea:
	•	rule_rare_asn = 1 if asn_rarity > some_threshold
	•	rule_rare_location = 1 if location_rarity > some_threshold
	3.	Stronger failure pattern
	•	You already have failure_streak_capped and burst_failure_count.
	•	Rule idea:
	•	rule_long_streak = 1 if failure_streak_capped == 5
	•	rule_burst_failures = 1 if burst_failure_count >= X
	4.	Country-based risk
	•	Some countries in EDA (e.g., small set with 50%+ attack rate) were much riskier.
	•	Rule idea:
	•	rule_high_risk_country = 1 if Country in {list_of_very_high_attack_rate_countries}

# Mapping rule_risk_score → rule_risk_band (low / medium / high)

In [ ]:
def map_rule_band(score: int) -> str:
    if score <= 1:
        return "low"
    elif score <= 3:
        return "medium"
    else:
        return "high"

df["rule_risk_band"] = df["rule_risk_score"].apply(map_rule_band)

print("Risk band counts:")
print(df["rule_risk_band"].value_counts())

Risk band counts:
rule_risk_band
low       191102
medium     92438
high       16460
Name: count, dtype: int64


So we found

	•	Low risk → ~63.7% of logins
	•	Medium risk → ~30.8%
	•	High risk → ~5.5%

# Checking Check attack rate per band using Is Attack IP

In [ ]:
label = "Is Attack IP"

print("\nAttack rate by rule_risk_band:")
print(df.groupby("rule_risk_band")[label].mean())

print("\nCrosstab of rule_risk_band vs label:")
print(pd.crosstab(df["rule_risk_band"], df[label]))


Attack rate by rule_risk_band:
rule_risk_band
high      0.183111
low       0.075337
medium    0.110788
Name: Is Attack IP, dtype: float64

Crosstab of rule_risk_band vs label:
Is Attack IP     False  True 
rule_risk_band               
high             13446   3014
low             176705  14397
medium           82197  10241


Overall attack rate in the dataset ≈ 9.2%.

So compared to the overall 9.2%:

	•	Low risk band (~7.5%)
	•	Slightly below global rate.
	•	This means the rules successfully push some attacks out of the low band, but not dramatically.
	•	Interpretation: low band is “mostly normal, but not clean enough to blindly trust”.
	•	Medium risk band (~11.1%)
	•	Above global rate.
	•	This is your main “suspicious but not crazy” zone – fits naturally with “step-up authentication”.
	•	High risk band (~18.3%)
	•	About 2× the global average.
	•	This is where the rules clearly concentrate higher risk logins.
	•	Great candidate for “block or heavily challenge”.

So the baseline does separate risk in the right direction:

low < overall < medium < high

That’s already a good sign.

	•	Risk is monotonic:
low → medium → high have increasing attack rates.
	•	High band is clearly high risk (~18.3% vs global 9.2%).
	•	Band distribution (63% low, 31% medium, 5.5% high) is realistic for:
	•	Allow most logins,
	•	Step-up a subset,
	•	Hard-check a small tail.

What’s not perfect (and expected at this stage) 🤏
	•	Low band still has 7.5% attack rate, not dramatically below global 9.2%.
	•	Means our rules do some filtering, but not super aggressive.
	•	Only 11% of all attacks fall into “high”.
	•	For a stronger baseline, you might want a higher fraction of attacks in medium+high bands.


# “We implemented a rule-based baseline v1 that uses five interpretable signals: unusual login time, off-hours access, new device, new ASN, and recent failure bursts. Each rule contributes a fixed number of points to a rule risk score, which is then mapped into low, medium, and high risk bands.

In our experiments on 300,000 login events, the overall attack rate is about 9.2%. In the low, medium, and high risk bands, the attack rates are approximately 7.5%, 11.1%, and 18.3% respectively. This shows that the rule-based baseline successfully concentrates higher-risk logins in the medium and high bands, but also that a significant portion of attacks still fall into the low band. This motivates the need for a learned intent classifier that can exploit richer temporal and contextual patterns than fixed rules.”

# v2 baseline

Why not using IP address -

* Super high cardinality There can be thousands/millions of distinct IPs. If you write rules like if IP == X then high risk, those rules won’t generalize to new IPs you’ve never seen.

* In this dataset you do know Is Attack IP. It would be trivial (but useless) to make a rule that says:
if this IP was ever marked attack before → risk=high.
That’s basically leaking the label into the feature. In the real world, you don’t know the future label yet.

	•	More meaningful abstractions exist


Instead of raw IP, we want:

	•	per-user IP behavior → is_new_ip_for_user
	•	IP’s network / ASN → ASN, new_asn_flag, asn_rarity
	•	geo context → Country/Region/City, location_rarity

“We avoid using raw IP addresses as rules, because they don’t generalize and basically act like a blacklist. Instead, we derive higher-level behavior features like ‘new ASN’, ‘new location’, and IP-based rarity. We also tested is_new_ip_for_user and found that, in this dataset, new IPs are less associated with attack IPs, likely because attackers re-use a small set of IPs while legitimate users move across networks. So we keep is_new_ip_for_user as an input feature for the ML model rather than using it as a simple risk-raising rule in the baseline.”



inspect attack rate for Device Type & Country

In [ ]:
import pandas as pd

label = "Is Attack IP"

# --- A) Device Type vs attack rate ---
print("Attack rate by Device Type:")
device_stats = (
    df.groupby("Device Type")[label]
      .agg(["count", "mean"])
      .rename(columns={"mean": "attack_rate"})
      .sort_values("attack_rate", ascending=False)
)
print(device_stats.head(10))  # top 10 risky device types

# --- B) Country vs attack rate (only if enough samples) ---
print("\nAttack rate by Country (only countries with >= 200 events):")
country_stats = (
    df.groupby("Country")[label]
      .agg(["count", "mean"])
      .rename(columns={"mean": "attack_rate"})
)
country_stats_filtered = country_stats[country_stats["count"] >= 200] \
    .sort_values("attack_rate", ascending=False)
print(country_stats_filtered.head(15))  # top 15 risky countries

Attack rate by Device Type:
              count  attack_rate
Device Type                     
bot             234     0.773504
unknown         167     0.323353
mobile       212206     0.117080
desktop       78276     0.029779
tablet         9117     0.026434

Attack rate by Country (only countries with >= 200 events):
         count  attack_rate
Country                    
ru         313     0.549521
cl         442     0.533937
ro        1733     0.466821
us       64532     0.310187
vn         225     0.288889
pk         203     0.266010
ph         474     0.175105
tr        1660     0.137952
id        4292     0.129077
au        3809     0.115516
pl       17398     0.098632
dk         342     0.087719
ca         669     0.074738
it         643     0.062208
nl         963     0.059190


Conclusion:
```
	•	bot → insanely risky (3/4 are attack IPs)
	•	unknown → moderate risk (about 1/3 attacks)
	•	mobile → slightly risky relative to global (~11.7% vs ~9.2%)
	•	desktop/tablet → low risk.

So it makes sense to treat:

Device Type in {bot, unknown} as risky device types.
```



```
ru, cl, ro → extremely high attack rates (≈ 0.47–0.55).
	•	us, vn, pk → clearly above global rate (~0.27–0.31).
	•	Others start dropping closer to / below overall 0.092.

So a simple, data-driven heuristic:

Treat countries with attack_rate >= 0.25 as high-risk countries.
```

rule_risky_device_type

	•	rule_high_risk_country

In [ ]:
label = "Is Attack IP"

# --- 1) Rule: risky device types (bot, unknown) ---
df["rule_risky_device_type"] = df["Device Type"].isin(["bot", "unknown"]).astype(int)

print("rule_risky_device_type value counts:")
print(df["rule_risky_device_type"].value_counts())

print("\nAttack rate by rule_risky_device_type:")
print(df.groupby("rule_risky_device_type")[label].mean())


# --- 2) Rule: high-risk countries ---

# You can either hardcode based on your stats:
# high_risk_countries = ["ru", "cl", "ro", "us", "vn", "pk"]

# Or compute them programmatically (optional):
country_stats = (
    df.groupby("Country")[label]
      .agg(["count", "mean"])
      .rename(columns={"mean": "attack_rate"})
)
high_risk_countries = country_stats[
    (country_stats["count"] >= 200) & (country_stats["attack_rate"] >= 0.25)
].index.tolist()
print("High-risk countries (>=200 events, attack_rate>=0.25):", high_risk_countries)

df["rule_high_risk_country"] = df["Country"].isin(high_risk_countries).astype(int)

print("\nrule_high_risk_country value counts:")
print(df["rule_high_risk_country"].value_counts())

print("\nAttack rate by rule_high_risk_country:")
print(df.groupby("rule_high_risk_country")[label].mean())

rule_risky_device_type value counts:
rule_risky_device_type
0    299599
1       401
Name: count, dtype: int64

Attack rate by rule_risky_device_type:
rule_risky_device_type
0    0.091512
1    0.586035
Name: Is Attack IP, dtype: float64
High-risk countries (>=200 events, attack_rate>=0.25): ['cl', 'pk', 'ro', 'ru', 'us', 'vn']

rule_high_risk_country value counts:
rule_high_risk_country
0    232552
1     67448
Name: count, dtype: int64

Attack rate by rule_high_risk_country:
rule_high_risk_country
0    0.027086
1    0.316585
Name: Is Attack IP, dtype: float64


	•	Risky device type (bot/unknown):
	•	Very rare (~0.13% of logins) but extremely dangerous.
	•	Perfect as a high-weight rule in v2.
	•	High-risk country (ru, cl, ro, us, vn, pk):
	•	~22.5% of all events.
	•	Attack rate is more than 10× higher than non-high-risk countries (31.7% vs 2.7%).
	•	This is a major risk driver.


Below are stats for v1

In [ ]:
def summarize_baseline(df, label_col, score_col, band_col):
    total = len(df)
    total_attacks = df[label_col].sum()
    global_rate = total_attacks / total

    print(f"Total events: {total}")
    print(f"Total attacks: {int(total_attacks)}")
    print(f"Global attack rate: {global_rate:.4f}")
    print("-" * 50)

    # --- Band-level stats ---
    band_stats = (
        df.groupby(band_col)[label_col]
          .agg(["count", "mean"])
          .rename(columns={"count": "events", "mean": "attack_rate"})
    )

    band_stats["events_pct"] = band_stats["events"] / total
    band_stats["attacks"] = (
        df.groupby(band_col)[label_col].sum().astype(int)
    )
    band_stats["attacks_pct_of_all"] = band_stats["attacks"] / total_attacks
    band_stats["attack_rate_lift"] = band_stats["attack_rate"] / global_rate

    print("Band-level stats:")
    display(band_stats.sort_index())

    # --- Rule-level stats ---
    rule_cols = [
        "rule_unusual_time",
        "rule_off_hours",
        "rule_new_device",
        "rule_new_asn",
        "rule_recent_failures",
        # later we’ll add:
        # "rule_risky_device_type",
        # "rule_high_risk_country",
    ]

    rows = []
    for col in rule_cols:
        trig = df[df[col] == 1]
        notrig = df[df[col] == 0]

        rows.append({
            "rule": col,
            "triggered_events": len(trig),
            "triggered_pct": len(trig) / total,
            "attack_rate_when_triggered": trig[label_col].mean(),
            "attack_rate_when_not_triggered": notrig[label_col].mean(),
        })

    rule_stats = pd.DataFrame(rows)
    print("\nRule-level stats:")
    display(rule_stats)

    return band_stats, rule_stats

In [ ]:
label_col = "Is Attack IP"
score_col = "rule_risk_score"      # will be rule_risk_score_v2 later
band_col = "rule_risk_band"        # will be rule_risk_band_v2 later

band_stats_v1, rule_stats_v1 = summarize_baseline(df, label_col, score_col, band_col)

Total events: 300000
Total attacks: 27652
Global attack rate: 0.0922
--------------------------------------------------
Band-level stats:


,events,attack_rate,events_pct,attacks,attacks_pct_of_all,attack_rate_lift
rule_risk_band,,,,,,
high,16460,0.183111,0.054867,3014,0.108998,1.986589
low,191102,0.075337,0.637007,14397,0.520650,0.817338
medium,92438,0.110788,0.308127,10241,0.370353,1.201950



Rule-level stats:


,rule,triggered_events,triggered_pct,attack_rate_when_triggered,attack_rate_when_not_triggered
0,rule_unusual_time,17524,0.058413,0.185916,0.086358
1,rule_off_hours,21774,0.072580,0.141637,0.088302
2,rule_new_device,115158,0.383860,0.113505,0.078884
3,rule_new_asn,98677,0.328923,0.110928,0.082981
4,rule_recent_failures,110221,0.367403,0.125366,0.072895


Band-level story


> The overall attack rate in the dataset is about 9.2%. In the low-risk band, the attack rate drops to 7.5%, which is slightly below the global rate (lift ≈ 0.82). This band covers about 63.7% of all logins and contains about 52% of all attack IPs.

> In the medium-risk band, the attack rate increases to 11.1% (lift ≈ 1.20), covering 30.8% of logins and about 37% of attacks.


> In the high-risk band, the attack rate reaches 18.3%, which is almost 2× the global average (lift ≈ 1.99), even though this band represents only 5.5% of all logins. It contains roughly 11% of all attack IPs.



Rule-level story

Unusual time (rule_unusual_time)

> Only ~5.8% of events trigger this rule, but their attack rate is 18.6% versus 8.6% when not triggered.

→ Very strong time-based anomaly signal.

	•	Off-hours (rule_off_hours)
	•	Triggers on ~7.3% of events; attack rate jumps to 14.2% vs 8.8% otherwise.

→ Off-hours access remains significantly riskier even when it’s not super rare for the user.

	•	New device (rule_new_device)
	•	Triggers in ~38.4% of events; attack rate 11.4% vs 7.9%.

→ Modest but consistent device-change risk.

	•	New ASN (rule_new_asn)
	•	Triggers in ~32.9% of events; attack rate 11.1% vs 8.3%.

→ Network path / provider change adds noticeable risk.

	•	Recent failure bursts (rule_recent_failures)
	•	Triggers in ~36.7% of events; attack rate 12.5% vs 7.3% when it doesn’t.

→ Recent failures are a strong indicator of malicious or problematic sessions.


The rule-based baseline successfully concentrates higher-risk logins into the high-risk band, where the attack rate is almost twice the global average (18.3% vs 9.2%), while the low-risk band shows a reduced attack rate of 7.5% over 63% of all logins.

All five rule signals show a higher attack rate when triggered compared to when they are inactive. In particular, unusual login times and recent failure bursts nearly double the attack rate, confirming that time-based anomalies and error patterns are strong intent signals.

 Compute rule_risk_score_v2

In [ ]:
# --- Rule-Based Risk Score V2 (with new rules) ---

df["rule_risk_score_v2"] = (
    2 * df["rule_unusual_time"]
    + 1 * df["rule_off_hours"]
    + 1 * df["rule_new_device"]
    + 1 * df["rule_new_asn"]
    + 1 * df["rule_recent_failures"]
    + 3 * df["rule_risky_device_type"]
    + 2 * df["rule_high_risk_country"]
)

print("rule_risk_score_v2 summary:")
print(df["rule_risk_score_v2"].describe())

print("\nCounts per rule_risk_score_v2:")
print(df["rule_risk_score_v2"].value_counts().sort_index())

rule_risk_score_v2 summary:
count    300000.000000
mean          1.723257
std           1.868419
min           0.000000
25%           0.000000
50%           1.000000
75%           3.000000
max          10.000000
Name: rule_risk_score_v2, dtype: float64

Counts per rule_risk_score_v2:
rule_risk_score_v2
0     128760
1      27444
2      34446
3      67576
4       4600
5      29283
6       2249
7       5538
8        101
10         3
Name: count, dtype: int64


	•	Score 0–2:

128,760 + 27,444 + 34,446 ≈ 190,650 events (~63.5%)

	•	Score 3–5:

67,576 + 4,600 + 29,283 ≈ 101,459 events (~33.8%)

	•	Score ≥ 6:

2,249 + 5,538 + 101 + 3 ≈ 7,891 events (~2.6%)

This already looks like a very natural band split:

	•	0–2 → low
	•	3–5 → medium
	•	6+ → high


Map v2 score → v2 bands

In [ ]:
def map_rule_band_v2(score: int) -> str:
    if score <= 2:
        return "low"
    elif score <= 5:
        return "medium"
    else:
        return "high"

df["rule_risk_band_v2"] = df["rule_risk_score_v2"].apply(map_rule_band_v2)

print("V2 Risk band counts:")
print(df["rule_risk_band_v2"].value_counts())

V2 Risk band counts:
rule_risk_band_v2
low       190650
medium    101459
high        7891
Name: count, dtype: int64


In [ ]:
def map_decision_from_band(band: str) -> str:
    if band == "low":
        return "ALLOW"
    elif band == "medium":
        return "STEP_UP"
    else:
        return "BLOCK"

df["rule_decision_v2"] = df["rule_risk_band_v2"].apply(map_decision_from_band)

print("Decision counts (v2):")
print(df["rule_decision_v2"].value_counts())

Decision counts (v2):
rule_decision_v2
ALLOW      190650
STEP_UP    101459
BLOCK        7891
Name: count, dtype: int64


Reuse the stats helper for v2

In [ ]:
def summarize_baseline(df, label_col, score_col, band_col):
    total = len(df)
    total_attacks = df[label_col].sum()
    global_rate = total_attacks / total

    print(f"Total events: {total}")
    print(f"Total attacks: {int(total_attacks)}")
    print(f"Global attack rate: {global_rate:.4f}")
    print("-" * 50)

    # --- Band-level stats ---
    band_stats = (
        df.groupby(band_col)[label_col]
          .agg(["count", "mean"])
          .rename(columns={"count": "events", "mean": "attack_rate"})
    )

    band_stats["events_pct"] = band_stats["events"] / total
    band_stats["attacks"] = (
        df.groupby(band_col)[label_col].sum().astype(int)
    )
    band_stats["attacks_pct_of_all"] = band_stats["attacks"] / total_attacks
    band_stats["attack_rate_lift"] = band_stats["attack_rate"] / global_rate

    print("Band-level stats:")
    display(band_stats.sort_index())

    # --- Rule-level stats ---
    rule_cols = [
        "rule_unusual_time",
        "rule_off_hours",
        "rule_new_device",
        "rule_new_asn",
        "rule_recent_failures",
        "rule_risky_device_type",
        "rule_high_risk_country",
    ]
    rows = []
    for col in rule_cols:
        trig = df[df[col] == 1]
        notrig = df[df[col] == 0]

        rows.append({
            "rule": col,
            "triggered_events": len(trig),
            "triggered_pct": len(trig) / total,
            "attack_rate_when_triggered": trig[label_col].mean(),
            "attack_rate_when_not_triggered": notrig[label_col].mean(),
        })

    rule_stats = pd.DataFrame(rows)
    print("\nRule-level stats:")
    display(rule_stats)

    return band_stats, rule_stats

In [ ]:
label_col = "Is Attack IP"
score_col_v2 = "rule_risk_score_v2"
band_col_v2 = "rule_risk_band_v2"

band_stats_v2, rule_stats_v2 = summarize_baseline(df, label_col, score_col_v2, band_col_v2)

Total events: 300000
Total attacks: 27652
Global attack rate: 0.0922
--------------------------------------------------
Band-level stats:


,events,attack_rate,events_pct,attacks,attacks_pct_of_all,attack_rate_lift
rule_risk_band_v2,,,,,,
high,7891,0.338107,0.026303,2668,0.096485,3.668162
low,190650,0.059785,0.635500,11398,0.412194,0.648614
medium,101459,0.133906,0.338197,13586,0.491321,1.452766



Rule-level stats:


,rule,triggered_events,triggered_pct,attack_rate_when_triggered,attack_rate_when_not_triggered
0,rule_unusual_time,17524,0.058413,0.185916,0.086358
1,rule_off_hours,21774,0.072580,0.141637,0.088302
2,rule_new_device,115158,0.383860,0.113505,0.078884
3,rule_new_asn,98677,0.328923,0.110928,0.082981
4,rule_recent_failures,110221,0.367403,0.125366,0.072895
5,rule_risky_device_type,401,0.001337,0.586035,0.091512
6,rule_high_risk_country,67448,0.224827,0.316585,0.027086


Justifiaction:

Risky device type (bot/unknown):
	•	Extremely rare (~0.13% of events) but highly dangerous — almost 60% of those logins come from attack IPs.
	•	When this rule doesn’t fire, the attack rate drops back to 9.15%, close to global.
High-risk countries (ru, cl, ro, ru, us, vn, pk):
	•	About 22.5% of all logins, but with an attack rate of 31.7%,
	•	Compared to only 2.7% in all other countries.
	•	That’s more than 10× difference → clearly a dominant risk factor.

This absolutely justifies adding them as high-weight rules in v2.

Compare to v1 (what you had before)

	•	v1 high band:
	•	events: 16,460 (~5.5%)
	•	attack_rate: 18.3% (lift ≈ 1.99)
	•	attacks captured: 3,014 (~10.9% of all attacks)
	•	v2 high band:
	•	events: 7,891 (~2.6%)
	•	attack_rate: 33.8% (lift ≈ 3.67)
	•	attacks captured: 2,668 (~9.6% of all attacks)

So:

	•	High band is now smaller but much more pure:
	•	from 18.3% → 33.8% attack rate
	•	i.e., almost 1 in 3 logins in high band is an attack.
	•	You sacrifice a tiny bit of coverage at the very top (10.9% → 9.6% of all attacks), but gain much higher precision.

Medium + High together (this is the key story)

	•	v1 medium+high attacks:
	•	10,241 (medium) + 3,014 (high) = 13,255
	•	≈ 48% of all attacks.
	•	v2 medium+high attacks:
	•	13,586 (medium) + 2,668 (high) = 16,254
	•	≈ 59% of all attacks.

👉 Huge improvement: your v2 baseline now concentrates ~59% of all attacks into 36% of the traffic (medium+high bands).


Low band got safer too

	•	v1 low band attack rate: 7.53% (lift ≈ 0.82)
	•	v2 low band attack rate: 5.98% (lift ≈ 0.65)

So:

	•	Low band is now even cleaner.
	•	You can say:
“If we treat the low band as ‘allow with no friction’, its attack rate is only about 65% of the global average.”

That’s a very nice argument for user convenience vs security trade-off.

# Rule-level interpretation (v2 uses the same rule stats)

rule_risky_device_type:

  ~0.13% of logins, attack rate ≈ 58.6% vs 9.15% otherwise

rule_high_risk_country:

  ~22.5% of logins, attack rate ≈ 31.7% vs 2.71% otherwise

# Conclusion

“We first built a simple rule-based baseline using time anomalies, device changes, ASN changes, and recent failure bursts. That baseline already separated risk into low, medium, and high bands, with the high band showing about twice the global attack rate.

In the second iteration, we incorporated two stronger contextual signals derived from our analysis: risky device types (such as ‘bot’ or ‘unknown’), and high-risk countries identified directly from the dataset. These rules are rare but highly predictive, so we assigned them higher weights in the risk score.

With the v2 rule baseline, the high-risk band now contains only 2.6% of all login attempts but has an attack rate of 33.8%, which is almost 4× the global average. At the same time, the low-risk band’s attack rate drops to about 6%, and the medium+high bands together capture nearly 59% of all attack IPs while covering only about 36% of the traffic. This shows that the enhanced rule-based system does a much better job of concentrating risky behavior, while keeping the low-risk band relatively clean and user-friendly.”

Now, if we look at each decision group:

	1.	ALLOW group (low risk)

	•	Contains about 190,650 logins.
	•	Only 5.9% of them are attacks.
	•	So this bucket is safer than average.

	2.	STEP_UP group (medium risk)

	•	About 101,459 logins.
	•	Around 13.4%  are attacks.
	•	So this is riskier than average, good place to ask for MFA.

	3.	BLOCK group (high risk)

	•	Only 7,891 logins.
	•	About 33.88 are attacks.
	•	So roughly 1 in 3 is an attack → very dangerous.

In [ ]:
label = "Is Attack IP"

print("\nAttack rate by decision (v2):")
print(df.groupby("rule_decision_v2")[label].mean())

print("\nCrosstab of decision vs label:")
print(pd.crosstab(df["rule_decision_v2"], df[label]))


Attack rate by decision (v2):
rule_decision_v2
ALLOW      0.059785
BLOCK      0.338107
STEP_UP    0.133906
Name: Is Attack IP, dtype: float64

Crosstab of decision vs label:
Is Attack IP       False  True 
rule_decision_v2               
ALLOW             179252  11398
BLOCK               5223   2668
STEP_UP            87873  13586


Predicted SAFE (ALLOW):

	•	Non-attack (True negative, TN): 179,252
	•	Attack (False negative, FN):   11,398

Predicted RISKY (STEP_UP + BLOCK):

Non-attack:

	•	STEP_UP: 87,873
	•	BLOCK:   5,223

→ False positives (FP) = 87,873 + 5,223 = 93,096

Attack:

	•	STEP_UP: 13,586
	•	BLOCK:   2,668
  
→ True positives (TP) = 13,586 + 2,668 = 16,254

🔴 False Positives (FP)

System says “risky” (STEP_UP/BLOCK), but it’s actually not an attack.

From above: 93,096 logins.

These are legitimate users that your rule engine is treating as suspicious:
	•	Good for security (you’re careful),
	•	But bad for user experience (they get extra friction or even blocked).

🔵 False Negatives (FN)

System says “safe” (ALLOW), but it is an attack.

From above: 11,398 logins.

These are attacks that slip through as low risk:
	•	Bad for security (missed detections),
	•	But zero friction for the attacker.

In [ ]:
import pandas as pd
import numpy as np

# 1. Binary prediction from rule decisions
# 1 = risky (STEP_UP or BLOCK), 0 = safe (ALLOW)
df["pred_attack"] = df["rule_decision_v2"].isin(["STEP_UP", "BLOCK"]).astype(int)

y_true = df["Is Attack IP"].astype(int)   # 1 = attack, 0 = normal
y_pred = df["pred_attack"]

# 2. Confusion matrix components
TP = int(((y_pred == 1) & (y_true == 1)).sum())
FP = int(((y_pred == 1) & (y_true == 0)).sum())
TN = int(((y_pred == 0) & (y_true == 0)).sum())
FN = int(((y_pred == 0) & (y_true == 1)).sum())

total = TP + FP + TN + FN

print("Confusion counts:")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}, Total: {total}\n")

# 3. Metrics (with safe division)
def safe_div(num, den):
    return num / den if den != 0 else np.nan

accuracy  = safe_div(TP + TN, total)
precision = safe_div(TP, TP + FP)         # Positive Predictive Value
recall    = safe_div(TP, TP + FN)         # True Positive Rate (TPR, sensitivity)
specificity = safe_div(TN, TN + FP)       # True Negative Rate (TNR)
fpr      = safe_div(FP, FP + TN)          # False Positive Rate
fnr      = safe_div(FN, FN + TP)          # False Negative Rate
f1       = safe_div(2 * precision * recall, precision + recall)
balanced_accuracy = (recall + specificity) / 2

# 4. Put metrics into a nice table
metrics = {
    "Metric": [
        "Accuracy",
        "Precision (PPV)",
        "Recall / TPR (Sensitivity)",
        "Specificity / TNR",
        "FPR (1 - Specificity)",
        "FNR (Miss Rate)",
        "F1-score",
        "Balanced Accuracy",
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        specificity,
        fpr,
        fnr,
        f1,
        balanced_accuracy,
    ],
}

metrics_df = pd.DataFrame(metrics)
metrics_df["Value"] = metrics_df["Value"].round(4)

metrics_df

Confusion counts:
TP: 16254, FP: 93096, TN: 179252, FN: 11398, Total: 300000



,Metric,Value
0,Accuracy,0.6517
1,Precision (PPV),0.1486
2,Recall / TPR (Sensitivity),0.5878
3,Specificity / TNR,0.6582
4,FPR (1 - Specificity),0.3418
5,FNR (Miss Rate),0.4122
6,F1-score,0.2373
7,Balanced Accuracy,0.6230


TP = 16,254  → attacks correctly flagged as risky

	•	FP = 93,096  → normal logins incorrectly flagged as risky
	•	TN = 179,252 → normal logins correctly treated as safe
	•	FN = 11,398  → attacks incorrectly treated as safe


In [ ]:
confusion_matrix_df = pd.DataFrame(
    {
        "Actual Normal (0)": [TN, FP],
        "Actual Attack (1)": [FN, TP],
    },
    index=["Predicted SAFE (0)", "Predicted RISKY (1)"],
)

confusion_matrix_df["Row Total"] = confusion_matrix_df.sum(axis=1)
confusion_matrix_df.loc["Column Total"] = confusion_matrix_df.sum(axis=0)

confusion_matrix_df

,Actual Normal (0),Actual Attack (1),Row Total
Predicted SAFE (0),179252,11398,190650
Predicted RISKY (1),93096,16254,109350
Column Total,272348,27652,300000


1️⃣ Accuracy (0.6517 → ~65%)

“How often is the system correct overall?”

	•	About 65% of all logins are classified correctly:
	•	either normal and treated as safe
	•	or attack and treated as risky

⚠️ In imbalanced problems like this (only ~9% attacks), accuracy is not the best metric, but okay to report.

⸻

2️⃣ Precision / PPV (0.1486 → ~14.9%)

“When we say a login is risky (STEP_UP/BLOCK), how often is it actually an attack?”

	•	Only about 15% of the logins we flag as risky are truly attacks.
	•	That means 85% of risky decisions are false positives (legit users being challenged/blocked).

👉 Interpretation:
	•	The system is paranoid: it flags a lot of logins as risky.
	•	Good for security, but high friction for users.

⸻

3️⃣ Recall / TPR / Sensitivity (0.5878 → ~58.8%)

“Out of all attack logins, how many do we catch as risky?”

	•	Your system correctly flags about 59% of attacks.
	•	The other 41% of attacks slip into ALLOW (these are FNs).

👉 Interpretation:
	•	Your baseline catches more than half of attacks.
	•	But still misses a large chunk, which is exactly why you want to improve using ML.

⸻

4️⃣ Specificity / TNR (0.6582 → ~65.8%)

“Out of all normal (non-attack) logins, how many do we correctly treat as safe?”

	•	About 66% of normal logins are correctly allowed.
	•	That means 34% of normal logins are incorrectly treated as risky (this is the FPR).

👉 Interpretation:
	•	One third of legitimate users get some extra friction (STEP_UP or BLOCK).
	•	This confirms the system is conservative and noisy.

⸻

5️⃣ FPR (0.3418 → ~34.2%)

False Positive Rate = ‘How many legit logins do we annoy?’

	•	Among all normal logins, 34% are flagged as risky.
	•	In cybersecurity, this is what leads to alert fatigue and frustrated users.

⸻

6️⃣ FNR (0.4122 → ~41.2%)

False Negative Rate = ‘How many attacks do we miss?’

	•	About 41% of attacks are treated as safe (they get ALLOW).
	•	This is dangerous from a security perspective.

⸻

7️⃣ F1-score (0.2373 → ~0.24)

“Balance between precision and recall for attacks.”

	•	F1 is low (~0.24), which reflects:
	•	okay recall (catching ~59% of attacks),
	•	but poor precision (lots of false alarms).

👉 Interpretation:
	•	The baseline finds many attacks, but not very cleanly (lots of noise).

⸻

8️⃣ Balanced Accuracy (0.6230 → ~62.3%)

“Average of: how good we are on attacks (recall) and how good we are on normal logins (specificity).”

\text{Balanced Accuracy} = \frac{TPR + TNR}{2} = \frac{0.5878 + 0.6582}{2} ≈ 0.623

	•	This metric treats attack and normal classes equally important.
	•	~62% is moderate performance.

  •	Time-based: is_off_hours, is_unusual_time_for_user, user_hour_std_q
  
	•	Behavior: new_device_flag, new_asn_flag, failures_last_5, burst_failure_count
	•	Rarity: location_rarity_q, device_type_rarity_q, asn_rarity_q
	•	Geo/device: high-risk countries, risky device types

In [ ]:
df.head()

,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,...,rule_new_asn,rule_recent_failures,rule_risk_score,rule_risk_band,rule_risky_device_type,rule_high_risk_country,rule_risk_score_v2,rule_risk_band_v2,rule_decision_v2,pred_attack
0,2020-02-06 17:10:54.364,-9223287066183308537,541.0,84.209.76.159,no,oslo county,oslo,41164,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...,Chrome 69.0.3497.17.19,...,0,0,0,low,0,0,0,low,ALLOW,0
1,2020-02-06 19:52:41.530,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0,0,0,low,0,0,0,low,ALLOW,0
2,2020-02-06 20:55:19.627,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0,0,0,low,0,0,0,low,ALLOW,0
3,2020-02-05 21:03:20.657,-9223200578825105501,541.0,79.161.56.83,no,vestfold og telemark,holmestrand,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Chrome Mobile 81.0.4044.2033,...,0,0,0,low,0,0,0,low,ALLOW,0
4,2020-02-06 19:12:29.501,-9223199305075633823,541.0,79.161.86.86,no,-,-,29695,Mozilla/5.0 (Linux; U; Android 13.0; i phone X...,Opera Mobile 52.1.2254,...,0,0,0,low,0,0,0,low,ALLOW,0


In [ ]:
print("Shape:", df.shape)
df.columns.tolist()

Shape: (300000, 70)


['Login Timestamp',
 'User ID',
 'Round-Trip Time [ms]',
 'IP Address',
 'Country',
 'Region',
 'City',
 'ASN',
 'User Agent String',
 'Browser Name and Version',
 'OS Name and Version',
 'Device Type',
 'Login Successful',
 'Is Attack IP',
 'Is Account Takeover',
 'browser',
 'os',
 'hour',
 'dayofweek',
 'is_new_device_for_user',
 'is_new_ip_for_user',
 'is_off_hours',
 'failed_login',
 'failures_last_5',
 'failure_streak',
 'failure_streak_capped',
 'location',
 'new_location_flag',
 'new_asn_flag',
 'device_fingerprint',
 'valid_device',
 'new_device_flag',
 'device_change_rate',
 'ts_sec',
 'delta_sec',
 'new_window',
 'window_id',
 'logins_5min',
 'failure_flag',
 'burst_failure_count',
 'failure_rate',
 'streak_reset',
 'streak_id',
 'failure_streak_length',
 'location_freq',
 'location_rarity',
 'device_type_freq',
 'device_type_rarity',
 'asn_freq',
 'asn_rarity',
 'location_rarity_q',
 'device_type_rarity_q',
 'asn_rarity_q',
 'user_offhour_rate',
 'is_unusual_time_for_user',

In [ ]:
# # Choose only the columns you actually want to carry forward.
# # For now, let's just save everything in df.
# # (You can refine later if needed.)

# save_path_parquet = "./data/rule_baseline__v1+v2.parquet"
# save_path_csv = "./data/rule_baseline__v1+v2.csv"

# # Parquet (preferred: smaller, preserves dtypes)
# df.to_parquet(save_path_parquet, index=False)

# # Optional CSV backup
# df.to_csv(save_path_csv, index=False)

# print("Saved feature dataset to:")
# print(" -", save_path_parquet)
# print(" -", save_path_csv)
# print("Shape:", df.shape)

Saved feature dataset to:
 - ./data/rule_baseline__v1+v2.parquet
 - ./data/rule_baseline__v1+v2.csv
Shape: (300000, 70)
